In [1]:
%%writefile prepare_data.py
import os
import argparse
import deepchem as dc
import pandas as pd
import logging
from datetime import datetime
from typing import List, Dict

task_dict = {'bbbp': ['p_np'], 
            'bace_classification': ['Class'],
            'clintox': ['FDA_APPROVED', 'CT_TOX'],
            'hiv': ['HIV_active'],
            'tox21': ['NR-AR', 'NR-AR-LBD', 'NR-AhR', 'NR-Aromatase', 'NR-ER', 'NR-ER-LBD',
                'NR-PPAR-gamma', 'SR-ARE', 'SR-ATAD5', 'SR-HSE', 'SR-MMP', 'SR-p53'],
            'sider': [
                'Hepatobiliary disorders', 'Metabolism and nutrition disorders',
                'Product issues', 'Eye disorders', 'Investigations',
                'Musculoskeletal and connective tissue disorders',
                'Gastrointestinal disorders', 'Social circumstances',
                'Immune system disorders', 'Reproductive system and breast disorders',
                'Neoplasms benign, malignant and unspecified (incl cysts and polyps)',
                'General disorders and administration site conditions',
                'Endocrine disorders', 'Surgical and medical procedures',
                'Vascular disorders', 'Blood and lymphatic system disorders',
                'Skin and subcutaneous tissue disorders',
                'Congenital, familial and genetic disorders', 'Infections and infestations',
                'Respiratory, thoracic and mediastinal disorders', 'Psychiatric disorders',
                'Renal and urinary disorders',
                'Pregnancy, puerperium and perinatal conditions',
                'Ear and labyrinth disorders', 'Cardiac disorders',
                'Nervous system disorders', 'Injury, poisoning and procedural complications'
            ],
            # esol is an alias for delaney dataset
            'delaney': ['measured log solubility in mols per litre'],
            'freesolv': ['y'],
            'lipo': ['exp'],
            'clearance': ['target'],
            'bace_regression': ['pIC50']
            }


def generate_deepchem_splits(dataset_names: List,
                             output_dir: str,
                             clean_smiles: bool=True,
                             max_smiles_len: int=200):
    """
    Generates scaffold splits for DeepChem datasets, optionally cleans them, and saves as CSVs.

    For each dataset name the corresponding 'dc.molnet.load_<name>' loader
    is called with a 'ScaffoldSplitter' and a 'DummyFeaturizer' so that
    the raw SMILES strings are preserved. When 'clean_smiles=True', any
    molecule whose SMILES length exceeds 'max_smiles_len' is removed before
    saving, which prevents excessively long sequences from dominating GPU
    memory during tokenisation. Results are written to
    '<output_dir>/<dataset_name>/{train,valid,test}.csv'.

    Parameters
    ----------
    dataset_names: List(str)
        List of dataset names.
    output_dir: str
        Folder to save the CSVs.
    clean_smiles: bool
        Whether to filter SMILES strings by max length.
    max_smiles_len: int
        Maximum allowed SMILES length if cleaning is enabled.

    Examples
    --------
    >>> import tempfile, os
    >>> from scripts.prepare_data import generate_deepchem_splits
    >>> with tempfile.TemporaryDirectory() as tmp:
    ...     generate_deepchem_splits(["bbbp"], output_dir=tmp, clean_smiles=True, max_smiles_len=200)
    ...     splits = os.listdir(os.path.join(tmp, "bbbp"))
    >>> sorted(splits)
    ['test.csv', 'train.csv', 'valid.csv']
    """
    os.makedirs(output_dir, exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_path = os.path.join(output_dir, f"deepchem_split_log_{timestamp}.log")

    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s [%(levelname)s] %(message)s",
        handlers=[logging.FileHandler(log_path), logging.StreamHandler()]
    )

    logging.info("Starting DeepChem dataset splitting...")

    for dataset_name in dataset_names:
        try:
            logging.info(f"Processing dataset: {dataset_name}")
            load_fn = getattr(dc.molnet, f'load_{dataset_name}')

            task_names, (train_set, valid_set, test_set), transformers = load_fn(
                featurizer=dc.feat.DummyFeaturizer(),
                transformers=[],
                splitter='scaffold',
                reload=False
            )

            dataset_dir = os.path.join(output_dir, dataset_name)
            os.makedirs(dataset_dir, exist_ok=True)

            split_names = ['train', 'valid', 'test']
            datasets = [train_set, valid_set, test_set]

            for split_name, split_set in zip(split_names, datasets):
                smiles_list = split_set.X
                y = split_set.y

                if clean_smiles:
                    mask = pd.Series(smiles_list).str.len() <= max_smiles_len
                    removed_count = (~mask).sum()
                    logging.info(f"{dataset_name} {split_name}: removed {removed_count} rows (SMILES > {max_smiles_len})")
                    smiles_list = [s for i, s in enumerate(smiles_list) if mask.iloc[i]]
                    y = y[mask.to_numpy()]

                data = {'smiles': smiles_list}
                data.update({task_names[i]: y[:, i] for i in range(len(task_names))})
                df = pd.DataFrame(data)
                df.to_csv(os.path.join(dataset_dir, f"{split_name}.csv"), index=False)
                logging.info(f"Saved {split_name} split to {dataset_dir}/{split_name}.csv")

        except AttributeError:
            logging.error(f"Dataset '{dataset_name}' not found in DeepChem.molnet. Skipping.")
        except Exception as e:
            logging.exception(f"Error while processing {dataset_name}: {str(e)}")

    logging.info("All datasets processed.")


def main():

    argparser = argparse.ArgumentParser()
    argparser.add_argument('--split_type',
                        type=str,
                        help='type of the splits to use for the datasets : molformer/deepchem',
                        default='deepchem')
    argparser.add_argument(
        '--datasets',
        type=str,
        help='comma-separated list of datasets to featurize',
        default='bbbp,bace_classification')
    argparser.add_argument('--data_dir',
                           type=str,
                           help='data dir of stored splits in csv format',
                           default='datasets')

    args = argparser.parse_args()

    datasets = args.datasets.split(',')

    if datasets is None:
        raise ValueError("Please provide a list of datasets to benchmark.")
    if not isinstance(datasets, list):
        raise ValueError("Datasets should be provided as a list.")
    if len(datasets) == 0:
        raise ValueError(
            "The list of datasets is empty. Please provide at least one dataset."
        )
    if args.split_type == 'deepchem':
        generate_deepchem_splits(dataset_names=datasets,
                                 output_dir=args.data_dir,
                                 clean_smiles=True,
                                 max_smiles_len=200)

if __name__ == "__main__":
    main()

Writing prepare_data.py


In [1]:
!python prepare_data.py \
--datasets bace_classification,clintox,sider,hiv,tox21,lipo,freesolv,delaney,bace_regression,clearance,bbbp,bace_classification \
--data_dir datasets

This module requires PyTorch to be installed.
Skipped loading some Pytorch utilities, missing a dependency. No module named 'torch'
No normalization for SPS. Feature removed!
No normalization for AvgIpc. Feature removed!
No normalization for NumAmideBonds. Feature removed!
No normalization for NumAtomStereoCenters. Feature removed!
No normalization for NumBridgeheadAtoms. Feature removed!
No normalization for NumHeterocycles. Feature removed!
No normalization for NumSpiroAtoms. Feature removed!
No normalization for NumUnspecifiedAtomStereoCenters. Feature removed!
No normalization for Phi. Feature removed!
Skipped loading some Tensorflow models, missing a dependency. No module named 'tensorflow'
Skipped loading some PyTorch models, missing a dependency. No module named 'torch'
No module named 'torch'
Skipped loading modules with pytorch-geometric dependency, missing a dependency. No module named 'torch'
Skipped loading modules with pytorch-lightning dependency, missing a dependency. No

In [2]:
import os
import pandas as pd

DATA_DIR = "/Users/harin/Documents/DFS/DFS/Claude_GPT_eval/datasets"
SPLITS = ["train", "valid", "test"]

rows = []
for name in sorted(os.listdir(DATA_DIR)):
    ds_dir = os.path.join(DATA_DIR, name)
    if not os.path.isdir(ds_dir):
        continue
    for split in SPLITS:
        df = pd.read_csv(os.path.join(ds_dir, f"{split}.csv"))
        lengths = df["smiles"].astype(str).str.len()
        rows.append({
            "dataset": name,
            "split": split,
            "n": len(lengths),
            "min": lengths.min(),
            "max": lengths.max(),
            "mean": round(lengths.mean(), 1),
            "median": lengths.median(),
            "p95": lengths.quantile(0.95),
        })

stats = pd.DataFrame(rows)
print(stats.to_string(index=False))


            dataset split     n  min  max  mean  median    p95
bace_classification train  1210   19  139  64.0    61.0  95.55
bace_classification valid   151   44  111  66.1    68.0  82.50
bace_classification  test   152   17  198  74.3    74.0 126.50
    bace_regression train  1210   19  139  64.0    61.0  95.55
    bace_regression valid   151   44  111  66.1    68.0  82.50
    bace_regression  test   152   17  198  74.3    74.0 126.50
               bbbp train  1640    3  199  48.7    44.0  98.00
               bbbp valid   203   16  200  68.8    60.0 154.70
               bbbp  test   189    8  170  49.5    45.0  92.00
          clearance train   668   17  196  52.6    52.0  79.00
          clearance valid    84   22  134  52.8    51.0  77.55
          clearance  test    84   21  122  50.5    49.0  83.55
            clintox train  1163    2  197  54.5    46.0 114.00
            clintox valid   142   24  195  70.9    59.0 158.95
            clintox  test   143   18  123  47.3    46.0

In [4]:
stats.to_csv('molnet_stats.csv')